# Day 29: Putting It All Together — Complete Training Toolkit

> **Goal:** Build a production-quality training framework that combines every technique you've learned, and benchmark all your models side by side.

---

## What You've Learned in 28 Days

| Phase | Days | Key Skills |
|---|---|---|
| Python & ML Basics | 1–3 | Python, NumPy, ML concepts |
| PyTorch Foundations | 4–9 | Tensors, autograd, training loop |
| Building Networks | 10–15 | Data loading, MLP, regularization |
| CNNs | 16–22 | Convolutions, architectures, visualization |
| Transformers & ViT | 22–28 | Attention, patch embedding, ViT |

Today we consolidate everything into a reusable toolkit.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import math
import time
from collections import defaultdict

torch.manual_seed(42)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

---
## 1. Reusable Training Framework

In [ ]:
class Trainer:
    """A reusable training framework that encapsulates best practices."""

    def __init__(self, model, device, lr=1e-3, weight_decay=0.01,
                 warmup_epochs=5, total_epochs=50, label_smoothing=0.0):
        self.model = model.to(device)
        self.device = device
        self.total_epochs = total_epochs

        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        self.optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

        # Warmup + cosine decay
        def lr_lambda(epoch):
            if epoch < warmup_epochs:
                return (epoch + 1) / (warmup_epochs + 1)
            progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
            return max(0.01, 0.5 * (1 + math.cos(math.pi * progress)))

        self.scheduler = optim.lr_scheduler.LambdaLR(self.optimizer, lr_lambda)
        self.history = defaultdict(list)
        self.best_val_acc = 0.0
        self.best_state = None

    def train_one_epoch(self, loader):
        self.model.train()
        total_loss, correct, total = 0, 0, 0
        for images, labels in loader:
            images, labels = images.to(self.device), labels.to(self.device)
            outputs = self.model(images)
            loss = self.loss_fn(outputs, labels)
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
        return total_loss / total, correct / total

    @torch.no_grad()
    def evaluate(self, loader):
        self.model.eval()
        total_loss, correct, total = 0, 0, 0
        for images, labels in loader:
            images, labels = images.to(self.device), labels.to(self.device)
            outputs = self.model(images)
            loss = self.loss_fn(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
        return total_loss / total, correct / total

    def fit(self, train_loader, val_loader, verbose=True):
        for epoch in range(self.total_epochs):
            t0 = time.time()
            tl, ta = self.train_one_epoch(train_loader)
            vl, va = self.evaluate(val_loader)
            lr_now = self.optimizer.param_groups[0]["lr"]
            self.scheduler.step()
            elapsed = time.time() - t0

            self.history["train_loss"].append(tl)
            self.history["val_loss"].append(vl)
            self.history["train_acc"].append(ta)
            self.history["val_acc"].append(va)
            self.history["lr"].append(lr_now)
            self.history["time"].append(elapsed)

            if va > self.best_val_acc:
                self.best_val_acc = va
                self.best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}

            if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
                print(f"  Ep {epoch+1:3d}/{self.total_epochs} | "
                      f"TrL {tl:.4f} | TrA {ta:.2%} | VL {vl:.4f} | VA {va:.2%} | "
                      f"LR {lr_now:.6f} | {elapsed:.1f}s")

        # Restore best model
        if self.best_state:
            self.model.load_state_dict(self.best_state)
            self.model.to(self.device)

        return self.history

    def test(self, test_loader):
        tl, ta = self.evaluate(test_loader)
        return ta

    def plot_history(self, title=""):
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        axes[0].plot(self.history["train_loss"], label="Train")
        axes[0].plot(self.history["val_loss"], label="Val")
        axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

        axes[1].plot(self.history["train_acc"], label="Train")
        axes[1].plot(self.history["val_acc"], label="Val")
        axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

        axes[2].plot(self.history["lr"])
        axes[2].set_title("Learning Rate"); axes[2].grid(True, alpha=0.3)

        for ax in axes: ax.set_xlabel("Epoch")
        plt.suptitle(title, fontsize=14)
        plt.tight_layout(); plt.show()

---
## 2. Model Zoo — All Architectures

In [ ]:
# === MLP ===
class MLP(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3*32*32, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes))

    def forward(self, x): return self.net(x)


# === Simple CNN ===
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, num_classes))

    def forward(self, x): return self.head(self.features(x))


# === ResNet-style CNN ===
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False), nn.BatchNorm2d(out_ch))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class ResNetSmall(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(3, 64, 3, 1, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU())
        self.layer1 = nn.Sequential(ResBlock(64, 64), ResBlock(64, 64))
        self.layer2 = nn.Sequential(ResBlock(64, 128, 2), ResBlock(128, 128))
        self.layer3 = nn.Sequential(ResBlock(128, 256, 2), ResBlock(256, 256))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(256, num_classes))

    def forward(self, x):
        x = self.conv1(x)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        return self.head(x)


# === ViT ===
class ViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, num_classes=10,
                 d_model=128, num_heads=4, num_layers=6, dropout=0.1):
        super().__init__()
        num_patches = (img_size // patch_size) ** 2
        self.patch_proj = nn.Conv2d(3, d_model, patch_size, patch_size)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, d_model) * 0.02)
        self.dropout = nn.Dropout(dropout)
        layer = nn.TransformerEncoderLayer(d_model, num_heads, d_model*4, dropout,
                                            "gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(),
                                   nn.Dropout(dropout), nn.Linear(d_model, num_classes))
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_proj(x).flatten(2).transpose(1, 2)
        x = torch.cat([self.cls_token.expand(B, -1, -1), x], dim=1)
        x = self.dropout(x + self.pos_embed)
        x = self.norm(self.encoder(x))
        return self.head(x[:, 0])


# Print summary
models_config = {
    "MLP": MLP(),
    "SimpleCNN": SimpleCNN(),
    "ResNet": ResNetSmall(),
    "ViT": ViT(),
}

print(f"{'Model':<12} {'Parameters':>12}")
print("-" * 26)
for name, m in models_config.items():
    params = sum(p.numel() for p in m.parameters())
    print(f"{name:<12} {params:>12,}")

---
## 3. Benchmark All Models

In [ ]:
# Data
CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    transforms.RandomErasing(p=0.2)])
test_transform = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])

full_train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
test_data  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)
train_data, val_data = random_split(full_train, [45000, 5000], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=128, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=128, shuffle=False)

### Preparing the Data

We use the same CIFAR-10 dataset with a consistent augmentation pipeline and train/val/test split. This is the same setup used across the entire course, so the results here are directly comparable to earlier days.

In [ ]:
# Train all models
EPOCHS = 30
all_results = {}

configs = {
    "MLP":       {"model_fn": MLP,       "lr": 1e-3, "wd": 1e-4, "warmup": 0, "ls": 0.0},
    "SimpleCNN": {"model_fn": SimpleCNN, "lr": 1e-3, "wd": 1e-4, "warmup": 0, "ls": 0.0},
    "ResNet":    {"model_fn": ResNetSmall,"lr": 1e-3, "wd": 1e-4, "warmup": 0, "ls": 0.0},
    "ViT":       {"model_fn": ViT,       "lr": 3e-4, "wd": 0.05, "warmup": 5, "ls": 0.1},
}

for name, cfg in configs.items():
    print(f"\n{'='*50}")
    print(f"Training {name}")
    print(f"{'='*50}")

    torch.manual_seed(42)
    model = cfg["model_fn"]()
    trainer = Trainer(model, device, lr=cfg["lr"], weight_decay=cfg["wd"],
                       warmup_epochs=cfg["warmup"], total_epochs=EPOCHS,
                       label_smoothing=cfg["ls"])
    history = trainer.fit(train_loader, val_loader)
    test_acc = trainer.test(test_loader)

    all_results[name] = {
        "history": history,
        "test_acc": test_acc,
        "params": sum(p.numel() for p in cfg["model_fn"]().parameters()),
        "avg_time": np.mean(history["time"]),
    }

    print(f"  ✅ {name} Test Accuracy: {test_acc:.2%}")

### Running the Benchmark

Below we train all four models using the `Trainer` class. Each model gets a tailored configuration:

- **MLP & CNNs:** Higher learning rate (1e-3), minimal regularization — these architectures are robust with small-scale data
- **ViT:** Lower learning rate (3e-4), warmup, heavy weight decay, and label smoothing — ViTs need these tricks to train well on small datasets

⚠️ **This cell trains 4 models sequentially — it will take a while!** The `Trainer` automatically saves the best checkpoint (based on validation accuracy) and restores it after training.

In [ ]:
# Grand comparison plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Val accuracy over training
for name, res in all_results.items():
    axes[0, 0].plot(res["history"]["val_acc"], label=name)
axes[0, 0].set_title("Validation Accuracy Over Training")
axes[0, 0].set_xlabel("Epoch"); axes[0, 0].set_ylabel("Accuracy")
axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

# Test accuracy bar chart
names = list(all_results.keys())
test_accs = [all_results[n]["test_acc"] for n in names]
colors = [f"C{i}" for i in range(len(names))]
axes[0, 1].barh(names, test_accs, color=colors)
axes[0, 1].set_xlabel("Test Accuracy")
axes[0, 1].set_title("Final Test Accuracy")
for i, (n, a) in enumerate(zip(names, test_accs)):
    axes[0, 1].text(a + 0.002, i, f"{a:.2%}", va="center")
axes[0, 1].grid(True, alpha=0.3, axis="x")

# Parameters
params = [all_results[n]["params"] for n in names]
axes[1, 0].barh(names, params, color=colors)
axes[1, 0].set_xlabel("Parameters")
axes[1, 0].set_title("Model Size")
for i, (n, p) in enumerate(zip(names, params)):
    axes[1, 0].text(p + 1000, i, f"{p:,}", va="center", fontsize=9)
axes[1, 0].grid(True, alpha=0.3, axis="x")

# Efficiency: accuracy vs speed
avg_times = [all_results[n]["avg_time"] for n in names]
axes[1, 1].scatter(avg_times, test_accs, s=np.array(params)/3000, color=colors, alpha=0.7)
for i, n in enumerate(names):
    axes[1, 1].annotate(n, (avg_times[i], test_accs[i]), textcoords="offset points",
                        xytext=(8, 0), fontsize=10)
axes[1, 1].set_xlabel("Avg Time per Epoch (s)")
axes[1, 1].set_ylabel("Test Accuracy")
axes[1, 1].set_title("Efficiency (bubble size = params)")
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle("Model Comparison on CIFAR-10", fontsize=16)
plt.tight_layout(); plt.show()

### Grand Comparison

The four-panel figure below gives a comprehensive view of all models:

1. **Top-left (Val Accuracy):** Learning curves — how fast each model reaches its peak. The MLP should plateau early while CNNs and ViTs continue improving.
2. **Top-right (Test Accuracy):** Final performance ranking — which architecture learned the best representations.
3. **Bottom-left (Parameters):** Model size — the ViT is typically the largest due to its dense attention layers.
4. **Bottom-right (Efficiency):** The "bang for your buck" chart — accuracy vs. training speed, with bubble size proportional to parameter count. Models in the **top-left** corner are ideal (fast and accurate).

In [ ]:
# Summary table
print(f"\n{'Model':<12} | {'Test Acc':>8} | {'Parameters':>12} | {'Avg Epoch':>10} | {'Acc/Param':>12}")
print("-" * 65)
for name in names:
    r = all_results[name]
    efficiency = r["test_acc"] / (r["params"] / 1e6)  # accuracy per million params
    print(f"{name:<12} | {r['test_acc']:>7.2%} | {r['params']:>12,} | {r['avg_time']:>9.1f}s | {efficiency:>10.1f}%/M")

### Efficiency Summary

The table below includes an **Acc/Param** metric (accuracy per million parameters). This measures how efficiently each model uses its parameters — a small model with high accuracy has great efficiency, while a large model with only marginally better accuracy has poor efficiency. This is important for deployment on edge devices or mobile phones.

---
## 4. The Training Cheat Sheet

Here's a reference card for training deep learning models:

### Architecture Selection
| Data Size | Recommended |
|---|---|
| < 1K images | Transfer learning (pretrained ResNet/ViT) |
| 1K–50K | CNN (ResNet-style) with augmentation |
| 50K+ | ViT or hybrid, with strong augmentation |

### Training Recipe
```python
# 1. Data: always augment
transforms.RandomHorizontalFlip()
transforms.RandomCrop(size, padding=4)
transforms.ColorJitter(...)
transforms.RandomErasing(p=0.1-0.25)

# 2. Model: use residual connections, layer/batch norm
# 3. Optimizer: AdamW with weight decay
optimizer = AdamW(params, lr=1e-3 to 3e-4, weight_decay=0.01-0.05)

# 4. LR Schedule: warmup + cosine decay
# 5. Regularization: dropout (0.1-0.3), weight decay, label smoothing
# 6. Gradient clipping: clip_grad_norm_(params, max_norm=1.0)
# 7. Always checkpoint the best model
```

---
## 🧪 Exercises

### Exercise 1: Ensemble
Take the 3 best models from the benchmark. Average their softmax outputs for each test image. Does the ensemble beat individual models?

### Exercise 2: Knowledge distillation
Use the ResNet as a "teacher" and a small MLP as a "student." Train the student to match the teacher's soft predictions (KL divergence loss).

### Exercise 3: Mixed precision
Use `torch.amp.autocast` and `GradScaler` for mixed-precision training. How much faster is it?

### Exercise 4: Custom dataset
Download a small image dataset (e.g., Flowers-102 or STL-10) and apply your training toolkit to it. Which model works best?

In [ ]:
# Exercises: Your code here

---
## 📝 Key Takeaways

| Lesson | Summary |
|---|---|
| **Reusable code** | Wrap training in a Trainer class — saves time and prevents bugs |
| **No free lunch** | Different architectures shine in different regimes |
| **CNN** | Best for small/medium data, fast training, strong locality bias |
| **ViT** | Best for large data, flexible, needs warmup & augmentation |
| **Regularization** | Always use: augmentation + dropout + weight decay |
| **LR schedule** | Warmup + cosine decay works universally |
| **Gradient clipping** | Prevents training instability |

**Tomorrow:** Capstone day — bring everything together for your final project!